# lp_phylo Usage Example

This notebook shows the manuscript-facing workflow: compute an averaged species distance table from input sequences, then optionally build a tree from that distance table.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "ladderpath.py").exists() else cwd.parent
if not (repo_root / "ladderpath.py").exists():
    raise RuntimeError("Run this notebook from the lp_phylo repository root or the examples folder.")

package_parent = repo_root.parent
if str(package_parent) not in sys.path:
    sys.path.insert(0, str(package_parent))

print(f"repo_root = {repo_root}")


## Compute the averaged distance table

`get_distance_table_average_from_seqs` reruns Ladderpath several times and averages the species-pair distances. The default distance method is `dice`; it is written explicitly here so the example is clear.


In [ ]:
from lp_phylo.ladderpath_tools.species_distance import get_distance_table_average_from_seqs

seqs = {
    "ATGGAGAAATCTCAA": 1,
    "TTTCGAATTCT": 1,
    "AGGATATTTATAAAT": 1,
    "ATCGATTTTGTT": 2,
    "CGGATGACTTCCT": 1,
}

species_info = {
    "species1": ("Morado", [-1, -2]),
    "species2": ("Musa_ornata", [-3, -4]),
    "species3": ("Kluai_Khai", [-5, -4]),
}

distance_table, name_map = get_distance_table_average_from_seqs(
    seqs,
    species_info,
    method="dice",
    n_runs=3,
    verbose=False,
)

distance_table, name_map


## Inspect the output

`distance_table` uses display labels as pair keys. `name_map` maps those labels back to species names.


In [ ]:
print("distance_table")
for pair, value in distance_table.items():
    print(f"  {pair}: {value:.6f}")

print("\nname_map")
for label, name in name_map.items():
    print(f"  {label}: {name}")


## Optional: build a tree

This step requires the tree-building dependencies used by `ladderpath_tools/phylo.py`. If they are unavailable in the local environment, the distance table above is still the main manuscript-facing output.


In [ ]:
try:
    from lp_phylo.ladderpath_tools.phylo import phylo

    tree = phylo(
        distance_table,
        name_map,
        method="upgma",
        show_plot=False,
        verbose=True,
    )
    print(tree)
except Exception as exc:
    print(f"Tree construction skipped: {exc}")
